# Question 11 - Vanilla GAN Discriminator Loss (MNIST)
$L_D = BCE(D(x_{real}), 1) + BCE(D(G(z)), 0)$

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

In [ ]:
bce = keras.losses.BinaryCrossentropy()

def compute_discriminator_loss(d_real_output, d_fake_output):
    real_loss = bce(tf.ones_like(d_real_output), d_real_output)
    fake_loss = bce(tf.zeros_like(d_fake_output), d_fake_output)
    return real_loss + fake_loss

In [ ]:
# Test cases
cases = {
    "perfect D (real=1, fake=0)": (tf.constant([[0.99]]), tf.constant([[0.01]])),
    "fooled D  (real=0, fake=1)": (tf.constant([[0.01]]), tf.constant([[0.99]])),
    "Nash eq.  (0.5, 0.5)":       (tf.constant([[0.5]]), tf.constant([[0.5]])),
}
for name, (r, f) in cases.items():
    print(f"{name}: loss = {compute_discriminator_loss(r, f).numpy():.4f}")
print("2*log(2) =", 2 * np.log(2))

In [ ]:
# Real use: a tiny discriminator on MNIST batch vs random fake images
(x_train, _), _ = keras.datasets.mnist.load_data()
real = (x_train[:64].astype("float32") - 127.5) / 127.5
fake = tf.random.uniform((64, 28, 28), -1, 1)

D = keras.Sequential([layers.Input((28, 28)), layers.Flatten(),
                      layers.Dense(128), layers.LeakyReLU(0.2),
                      layers.Dense(1, activation='sigmoid')])
print("D loss on MNIST batch:", compute_discriminator_loss(D(real), D(fake)).numpy())

### 2. Ideal predictions at Nash equilibrium
When $p_g = p_{data}$, the optimal discriminator is $D^*(x) = \frac{p_{data}}{p_{data}+p_g} = 0.5$.

So **$D(x_{real}) = 0.5$ and $D(G(z)) = 0.5$**: D cannot tell real from fake, and $L_D = 2\log 2 \approx 1.386$.